In [8]:
import os
import re
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/biancaroman/pmb-dosare2/pmb_dosare2.csv')

def get_dpg_from_pdf_name(pdf_name):
    match = re.match(r'[\d]+_(\d+)_', str(pdf_name))
    if match:
        return int(match.group(1))
    return None

def get_dpg_from_solutie(solutie_str):
    match = re.search(r'DPG:\s*(\d+)', str(solutie_str))
    if match:
        return int(match.group(1))
    return None

def get_solutie_text_from_solutie(solutie_str):
    match = re.search(r'DPG:\s*\d+,\s*Dat[aă]:\s*[\d\-]+,\s*(.+)', str(solutie_str))
    if match:
        return match.group(1).strip()
    return None

def parse_istorie_acte(istorie_str):
    if pd.isna(istorie_str):
        return []
    entries = re.findall(
        r'DPG:\s*(\d+)\s+Dat[aă]:\s*[\d\-]+\s+([^\n]+?)(?=\s+DPG:|$)',
        str(istorie_str)
    )
    return [(int(dpg), solutie.strip()) for dpg, solutie in entries]

def clasificare_noua(val):
    if pd.isna(val) or val == 'NONE' or val is None:
        return 'NONE'
    val_lower = str(val).lower()
    has_restituire = bool(re.search(r'restit|rest\.', val_lower))
    has_respingere = bool(re.search(r'resp|\brn\b', val_lower))
    has_revocare = bool(re.search(r'revoc|anul', val_lower))
    has_declinare = bool(re.search(r'declin|decl|transmis|transmitere|retur|returnat|intors|djcl', val_lower))
    has_compensare = bool(re.search(r'\bmre\b|\bmcp\b|masuri|compens|despagubire', val_lower))
    if has_restituire:
        return 'Restituire'
    elif has_compensare:
        return 'Compensare/Despagubiri'
    elif has_respingere:
        return 'Respins/Negativ'
    elif has_revocare:
        return 'Revocare/Anulare'
    elif has_declinare:
        return 'Declinare/Transfer'
    return 'NONE'

def get_solutie_pentru_pdf(row):
    pdf_name = row['Pdf_nume']
    if pd.isna(pdf_name) or pdf_name == '':
        return None, None
    dpg_pdf = get_dpg_from_pdf_name(pdf_name)
    if dpg_pdf is None:
        return None, None
    dpg_final = get_dpg_from_solutie(row['Soluție'])
    if dpg_final is not None and dpg_pdf == dpg_final:
        solutie_text = get_solutie_text_from_solutie(row['Soluție'])
        return solutie_text, clasificare_noua(solutie_text)
    istorie = parse_istorie_acte(row['Istorie acte'])
    for dpg, solutie_text in istorie:
        if dpg == dpg_pdf:
            return solutie_text, clasificare_noua(solutie_text)
    return None, None

df_cu_pdf = df[df['Pdf_nume'].notna() & (df['Pdf_nume'] != '')].copy()
df_cu_pdf[['solutie_pdf_text', 'solutie_pdf_grup']] = df_cu_pdf.apply(
    lambda row: pd.Series(get_solutie_pentru_pdf(row)), axis=1
)
df_cu_pdf['Pdf_nume'] = df_cu_pdf['Pdf_nume'].str.split(';')
df_cu_pdf = df_cu_pdf.explode('Pdf_nume')
df_cu_pdf['Pdf_nume'] = df_cu_pdf['Pdf_nume'].str.strip()
df_cu_pdf = df_cu_pdf.drop_duplicates(subset='Pdf_nume').reset_index(drop=True)

txt_dir = '/kaggle/input/datasets/biancaroman/text-azure/pdf_text_azure'
files = [f for f in os.listdir(txt_dir) if f.endswith('.txt')]
pdf_disponibile = set(f.replace('.txt', '.pdf') for f in files)
df_cu_pdf['are_txt'] = df_cu_pdf['Pdf_nume'].isin(pdf_disponibile)

print("TOATE PDF-URILE UNICE DIN CSV")
print(f"Total: {len(df_cu_pdf)}")
print(df_cu_pdf['solutie_pdf_grup'].value_counts(dropna=False))

print("\nCATE AU TXT DISPONIBIL, PE SOLUTIE")
print(f"Total cu txt: {df_cu_pdf['are_txt'].sum()}")
print(df_cu_pdf[df_cu_pdf['are_txt']]['solutie_pdf_grup'].value_counts(dropna=False))

print("\nCATE NU AU TXT, PE SOLUTIE ")
print(f"Total fara txt: {(~df_cu_pdf['are_txt']).sum()}")
print(df_cu_pdf[~df_cu_pdf['are_txt']]['solutie_pdf_grup'].value_counts(dropna=False))

print("\nPDF-URI IN TXT_AZURE DAR NU IN CSV")
pdf_in_csv = set(df_cu_pdf['Pdf_nume'])
extra = pdf_disponibile - pdf_in_csv
print(f"Total extra: {len(extra)}")
print(list(extra)[:10])

/tmp/ipykernel_55/3283816030.py:5: DtypeWarning: Columns (17,18,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/kaggle/input/datasets/biancaroman/pmb-dosare2/pmb_dosare2.csv')


TOATE PDF-URILE UNICE DIN CSV
Total: 7471
solutie_pdf_grup
Restituire                2552
Respins/Negativ           2442
Compensare/Despagubiri    2432
Declinare/Transfer          24
Revocare/Anulare            20
NONE                         1
Name: count, dtype: int64

CATE AU TXT DISPONIBIL, PE SOLUTIE
Total cu txt: 4020
solutie_pdf_grup
Restituire                1795
Respins/Negativ           1242
Compensare/Despagubiri     951
Revocare/Anulare            20
Declinare/Transfer          12
Name: count, dtype: int64

CATE NU AU TXT, PE SOLUTIE 
Total fara txt: 3451
solutie_pdf_grup
Compensare/Despagubiri    1481
Respins/Negativ           1200
Restituire                 757
Declinare/Transfer          12
NONE                         1
Name: count, dtype: int64

PDF-URI IN TXT_AZURE DAR NU IN CSV
Total extra: 0
[]


In [12]:
import os
import re
import numpy as np
import pandas as pd
import torch
import umap
import hdbscan
import plotly.graph_objects as go
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

df = pd.read_csv('/kaggle/input/datasets/biancaroman/pmb-dosare2/pmb_dosare2.csv')

def get_dpg_from_pdf_name(pdf_name):
    match = re.match(r'[\d]+_(\d+)_', str(pdf_name))
    return int(match.group(1)) if match else None

def get_dpg_from_solutie(solutie_str):
    match = re.search(r'DPG:\s*(\d+)', str(solutie_str))
    return int(match.group(1)) if match else None

def get_solutie_text_from_solutie(solutie_str):
    match = re.search(r'DPG:\s*\d+,\s*Dat[aă]:\s*[\d\-]+,\s*(.+)', str(solutie_str))
    return match.group(1).strip() if match else None

def parse_istorie_acte(istorie_str):
    if pd.isna(istorie_str):
        return []
    entries = re.findall(
        r'DPG:\s*(\d+)\s+Dat[aă]:\s*[\d\-]+\s+([^\n]+?)(?=\s+DPG:|$)',
        str(istorie_str)
    )
    return [(int(dpg), solutie.strip()) for dpg, solutie in entries]

def clasificare_noua(val):
    if pd.isna(val) or val == 'NONE' or val is None:
        return 'NONE'
    val_lower = str(val).lower()
    if re.search(r'restit|rest\.', val_lower): return 'Restituire'
    elif re.search(r'\bmre\b|\bmcp\b|masuri|compens|despagubire', val_lower): return 'Compensare/Despagubiri'
    elif re.search(r'resp|\brn\b', val_lower): return 'Respins/Negativ'
    elif re.search(r'revoc|anul', val_lower): return 'Revocare/Anulare'
    elif re.search(r'declin|decl|transmis|transmitere|retur|returnat|intors|djcl', val_lower): return 'Declinare/Transfer'
    return 'NONE'

def get_solutie_pentru_pdf(row):
    pdf_name = row['Pdf_nume']
    if pd.isna(pdf_name) or pdf_name == '':
        return None, None
    dpg_pdf = get_dpg_from_pdf_name(pdf_name)
    if dpg_pdf is None:
        return None, None
    dpg_final = get_dpg_from_solutie(row['Soluție'])
    if dpg_final is not None and dpg_pdf == dpg_final:
        solutie_text = get_solutie_text_from_solutie(row['Soluție'])
        return solutie_text, clasificare_noua(solutie_text)
    for dpg, solutie_text in parse_istorie_acte(row['Istorie acte']):
        if dpg == dpg_pdf:
            return solutie_text, clasificare_noua(solutie_text)
    return None, None

df_cu_pdf = df[df['Pdf_nume'].notna() & (df['Pdf_nume'] != '')].copy()
df_cu_pdf[['solutie_pdf_text', 'solutie_pdf_grup']] = df_cu_pdf.apply(
    lambda row: pd.Series(get_solutie_pentru_pdf(row)), axis=1
)
df_cu_pdf['Pdf_nume'] = df_cu_pdf['Pdf_nume'].str.split(';')
df_cu_pdf = df_cu_pdf.explode('Pdf_nume')
df_cu_pdf['Pdf_nume'] = df_cu_pdf['Pdf_nume'].str.strip()
df_cu_pdf = df_cu_pdf.drop_duplicates(subset='Pdf_nume').reset_index(drop=True)

txt_dir = '/kaggle/input/datasets/biancaroman/text-azure/pdf_text_azure'
pdf_disponibile = set(f.replace('.txt', '.pdf') for f in os.listdir(txt_dir) if f.endswith('.txt'))
df_cu_pdf['are_txt'] = df_cu_pdf['Pdf_nume'].isin(pdf_disponibile)

SOLUTII_VALIDE = {'Restituire', 'Compensare/Despagubiri', 'Respins/Negativ'}
df_final = df_cu_pdf[
    df_cu_pdf['are_txt'] &
    df_cu_pdf['solutie_pdf_grup'].isin(SOLUTII_VALIDE)
].reset_index(drop=True)

print(f"Total documente: {len(df_final)}")
print(df_final['solutie_pdf_grup'].value_counts())


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

bert_tokenizer = AutoTokenizer.from_pretrained('dumitrescustefan/bert-base-romanian-cased-v1')
bert_model = AutoModel.from_pretrained('dumitrescustefan/bert-base-romanian-cased-v1')
bert_model = bert_model.to(device)
bert_model.eval()

ner_tokenizer = AutoTokenizer.from_pretrained('BiancaRoman28/xlm-roberta-fine-tuned-pmb')
ner_model = AutoModelForTokenClassification.from_pretrained('BiancaRoman28/xlm-roberta-fine-tuned-pmb')
ner_model = ner_model.to(device)
ner_model.eval()
id2label = ner_model.config.id2label


SIGNATURE_PATTERN = re.compile(
    r'(Copia dispozi[tț]iei se va comunica|Dispozi[tț]ia se comunica)',
    re.IGNORECASE
)
CORRUPT_PATTERN = re.compile(r'^[A-Za-z0-9+/]{50,}={0,2}$')

def is_corrupt(text):
    first_line = text.strip().split('\n')[0].strip()
    return bool(CORRUPT_PATTERN.match(first_line)) or len(text.strip()) < 100

def remove_signatures(text):
    match = SIGNATURE_PATTERN.search(text)
    return text[:match.start()].strip() if match else text.strip()

SOLUTIE_PATTERN = re.compile(
    r'(SE\s+RESTITUIE[\s,]+IN\s+NATURA|SE\s+RESTITUIE\b|'
    r'restituie[\s,]+in\s+natura|restituie\b|'
    r'restituire\s+in\s+natura|restituirea\s+in\s+natura|'
    r'SE\s+RESPINGE\b|Se\s+respinge\b|RESPINGE\b|respinge\b|'
    r'SE\s+ACORDA\s+MASURI\s+REPARATORII(\s+PRIN\s+ECHIVALENT|\s+IN\s+ECHIVALENT)?|'
    r'Se\s+acorda\s+masuri\s+reparatorii(\s+prin\s+echivalent|\s+in\s+echivalent)?|'
    r'MASURI\s+REPA[RT]ATORII(\s+PRIN\s+ECHIVALENT|\s+IN\s+ECHIVALENT)?|'
    r'masuri\s+repa[rt]atorii(\s+prin\s+echivalent|\s+in\s+echivalent)?|'
    r'acordarea\s+de\s+masuri\s+reparatorii|'
    r'SE\s+ATRIBUIE\s+IN\s+FOLOSINTA\s+SPECIALA|'
    r'despagubiri\s+prin\s+echivalent|DESPAGUBIRI\s+PRIN\s+ECHIVALENT)',
    re.IGNORECASE
)

def remove_solutie_keywords(text):
    return SOLUTIE_PATTERN.sub('[SOLUTIE]', text)

CNP_PATTERN = re.compile(r'\b[1-9]\d{12}\b')

def anonymize_text(text):
    text = CNP_PATTERN.sub('[CNP]', text)
    tokens = ner_tokenizer.encode(text, add_special_tokens=False)
    token_labels = {}
    for offset in range(0, len(tokens), 510):
        chunk = tokens[offset:offset + 510]
        input_ids = torch.tensor([[ner_tokenizer.cls_token_id] + chunk + [ner_tokenizer.sep_token_id]]).to(device)
        attention_mask = torch.ones_like(input_ids).to(device)
        with torch.no_grad():
            output = ner_model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = output.logits[0].argmax(dim=-1).cpu().numpy()
        for i, pred in enumerate(predictions[1:-1]):
            token_labels[offset + i] = id2label[pred]
    all_tokens = ner_tokenizer.convert_ids_to_tokens(tokens)
    result = []
    i = 0
    while i < len(all_tokens):
        label = token_labels.get(i, 'O')
        if label.startswith('B-'):
            entity_type = label[2:]
            i += 1
            while i < len(all_tokens) and token_labels.get(i, 'O') == f'I-{entity_type}':
                i += 1
            result.append(f'[{entity_type}]')
        else:
            result.append(all_tokens[i])
            i += 1
    return ner_tokenizer.convert_tokens_to_string(result)

def mean_pooling_chunks(text, chunk_size=512, overlap=64):
    tokens = bert_tokenizer.encode(text, add_special_tokens=False)
    stride = chunk_size - overlap - 2
    chunks = []
    for i in range(0, max(1, len(tokens)), stride):
        chunk = tokens[i:i + chunk_size - 2]
        chunk = [bert_tokenizer.cls_token_id] + chunk + [bert_tokenizer.sep_token_id]
        chunks.append(chunk)
        if i + chunk_size - 2 >= len(tokens):
            break
    chunk_embeddings = []
    for chunk in chunks:
        input_ids = torch.tensor([chunk]).to(device)
        attention_mask = torch.ones_like(input_ids).to(device)
        with torch.no_grad():
            output = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        embedding = output.last_hidden_state[0].mean(dim=0).cpu().numpy()
        chunk_embeddings.append(embedding)
    result = np.mean(chunk_embeddings, axis=0)
    result = result / np.linalg.norm(result)
    return result

texts_raw = {}
for f in os.listdir(txt_dir):
    if not f.endswith('.txt'):
        continue
    with open(os.path.join(txt_dir, f), 'r', encoding='utf-8', errors='ignore') as fp:
        text = fp.read()
    if is_corrupt(text):
        continue
    texts_raw[f.replace('.txt', '.pdf')] = remove_signatures(text)

df_final['text_v1'] = df_final['Pdf_nume'].map(texts_raw)
df_final['text_v2'] = df_final['text_v1'].apply(
    lambda t: remove_solutie_keywords(t) if pd.notna(t) else t
)

print("Anonimizez V3")
df_final['text_v3'] = None
for idx, row in tqdm(df_final.iterrows(), total=len(df_final)):
    try:
        df_final.at[idx, 'text_v3'] = anonymize_text(row['text_v2'])
    except:
        df_final.at[idx, 'text_v3'] = row['text_v2']

df_final = df_final[df_final['text_v1'].notna()].reset_index(drop=True)
print(f"Dupa curatare: {len(df_final)}")

#CLUSTER
le = LabelEncoder()
labels_true = le.fit_transform(df_final['solutie_pdf_grup'])

def compute_embeddings(texts, label):
    print(f"Embeddings {label}...")
    embeddings = np.array([mean_pooling_chunks(t) for t in tqdm(texts)])
    np.save(f'/kaggle/working/embeddings_{label}.npy', embeddings)
    print(f"Shape: {embeddings.shape}")
    return embeddings

def run_pipeline(embeddings, label):
    print(f"UMAP {label}...")
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
    umap_emb = reducer.fit_transform(embeddings)
    np.save(f'/kaggle/working/umap_{label}.npy', umap_emb)

    print(f"HDBSCAN {label}...")
    clusterer = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=10,
                                  metric='euclidean', cluster_selection_method='eom')
    cluster_labels = clusterer.fit_predict(umap_emb)

    mask = cluster_labels >= 0
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    noise_pct = (cluster_labels == -1).sum() / len(cluster_labels) * 100
    sil = silhouette_score(umap_emb[mask], cluster_labels[mask]) if mask.sum() > 1 else 0
    ari = adjusted_rand_score(labels_true[mask], cluster_labels[mask])
    nmi = normalized_mutual_info_score(labels_true[mask], cluster_labels[mask])

    print(f"  Clustere: {n_clusters} | Noise: {noise_pct:.1f}% | Sil: {sil:.4f} | ARI: {ari:.4f} | NMI: {nmi:.4f}")

    return umap_emb, cluster_labels, {
        'Varianta': label,
        'Clustere': n_clusters,
        'Noise %': round(noise_pct, 1),
        'Silhouette': round(sil, 4),
        'ARI': round(ari, 4),
        'NMI': round(nmi, 4)
    }

def plot_plotly(umap_emb, cluster_labels, title, filename):
    CULORI = {
        'Restituire': '#2ca02c',
        'Compensare/Despagubiri': '#1f77b4',
        'Respins/Negativ': '#d62728'
    }
    fig = go.Figure()
    for solutie, culoare in CULORI.items():
        mask = df_final['solutie_pdf_grup'] == solutie
        idx = np.where(mask)[0]
        fig.add_trace(go.Scatter(
            x=umap_emb[idx, 0],
            y=umap_emb[idx, 1],
            mode='markers',
            name=solutie,
            marker=dict(color=culoare, size=5, opacity=0.6),
            customdata=np.stack([
                df_final.loc[mask, 'Pdf_nume'].values,
                df_final.loc[mask, 'solutie_pdf_grup'].values,
                [f'Cluster {l}' if l >= 0 else 'Noise' for l in cluster_labels[idx]],
                df_final.loc[mask, 'text_v1'].str[:200].str.replace('\n', ' ').values
            ], axis=-1),
            hovertemplate=(
                '<b>%{customdata[0]}</b><br>'
                'Solutie: %{customdata[1]}<br>'
                'Cluster: %{customdata[2]}<br>'
                # 'Text: %{customdata[3]}<br>'
                '<extra></extra>'
            )
        ))
    fig.update_layout(
        title=title, xaxis_title='UMAP 1', yaxis_title='UMAP 2',
        width=1100, height=700, legend_title='Soluție', template='plotly_white'
    )
    fig.write_html(f'/kaggle/working/{filename}.html')
    print(f"Salvat: {filename}.html")

results = []

emb_v1 = compute_embeddings(df_final['text_v1'].tolist(), 'v1')
umap_v1, labels_v1, m1 = run_pipeline(emb_v1, 'v1')
plot_plotly(umap_v1, labels_v1, 'V1: Text complet', 'plotly_v1')
df_final['cluster_v1'] = labels_v1
results.append(m1)

emb_v2 = compute_embeddings(df_final['text_v2'].tolist(), 'v2')
umap_v2, labels_v2, m2 = run_pipeline(emb_v2, 'v2')
plot_plotly(umap_v2, labels_v2, 'V2: Fara cuvinte solutie', 'plotly_v2')
df_final['cluster_v2'] = labels_v2
results.append(m2)

emb_v3 = compute_embeddings(df_final['text_v3'].tolist(), 'v3')
umap_v3, labels_v3, m3 = run_pipeline(emb_v3, 'v3')
plot_plotly(umap_v3, labels_v3, 'V3: Anonimizat + fara solutie', 'plotly_v3')
df_final['cluster_v3'] = labels_v3
results.append(m3)

df_metrics = pd.DataFrame(results)
print("\nTABEL METRICI")
print(df_metrics.to_string(index=False))
df_metrics.to_csv('/kaggle/working/metrici_clustering.csv', index=False)
df_final.to_csv('/kaggle/working/df_final_clustering.csv', index=False)

2026-04-05 18:47:00.015887: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775414820.234308      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775414820.304616      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775414820.812369      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775414820.812410      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775414820.812412      55 computation_placer.cc:177] computation placer alr

Total documente: 3988
solutie_pdf_grup
Restituire                1795
Respins/Negativ           1242
Compensare/Despagubiri     951
Name: count, dtype: int64


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Shape: (3961, 768)
UMAP v1...
HDBSCAN v1...
  Clustere: 18 | Noise: 1.3% | Sil: 0.3678 | ARI: 0.2339 | NMI: 0.4064
Salvat: plotly_v1.html
Embeddings v2...


100%|██████████| 3961/3961 [04:09<00:00, 15.89it/s]
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Shape: (3961, 768)
UMAP v2...
HDBSCAN v2...
  Clustere: 28 | Noise: 9.6% | Sil: 0.6288 | ARI: 0.1436 | NMI: 0.3564
Salvat: plotly_v2.html
Embeddings v3...


100%|██████████| 3961/3961 [03:49<00:00, 17.24it/s]
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Shape: (3961, 768)
UMAP v3...
HDBSCAN v3...
  Clustere: 28 | Noise: 4.5% | Sil: 0.5536 | ARI: 0.1523 | NMI: 0.3703
Salvat: plotly_v3.html

TABEL METRICI
Varianta  Clustere  Noise %  Silhouette    ARI    NMI
      v1        18      1.3      0.3678 0.2339 0.4064
      v2        28      9.6      0.6288 0.1436 0.3564
      v3        28      4.5      0.5536 0.1523 0.3703


In [13]:
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def compute_intra_cluster_cosine(embeddings, cluster_labels):
    scores = []
    for cl in set(cluster_labels):
        if cl == -1:
            continue
        mask = cluster_labels == cl
        emb_cl = embeddings[mask]
        if len(emb_cl) < 2:
            continue
        centroid = emb_cl.mean(axis=0)
        centroid = centroid / np.linalg.norm(centroid)
        sims = cosine_similarity(emb_cl, centroid.reshape(1, -1)).flatten()
        scores.append(sims.mean())
    return np.mean(scores)

def compute_all_metrics(embeddings, umap_emb, cluster_labels, true_labels, label):
    mask = cluster_labels >= 0
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    noise_pct = (cluster_labels == -1).sum() / len(cluster_labels) * 100
    
    sil = silhouette_score(umap_emb[mask], cluster_labels[mask]) if mask.sum() > 1 else 0
    ari = adjusted_rand_score(true_labels[mask], cluster_labels[mask])
    nmi = normalized_mutual_info_score(true_labels[mask], cluster_labels[mask])
    hom = homogeneity_score(true_labels[mask], cluster_labels[mask])
    comp = completeness_score(true_labels[mask], cluster_labels[mask])
    vmeas = v_measure_score(true_labels[mask], cluster_labels[mask])
    cos_intra = compute_intra_cluster_cosine(embeddings, cluster_labels)
    
    print(f"Varianta: {label}")
    print(f"  Clustere:          {n_clusters}")
    print(f"  Noise %:           {noise_pct:.1f}%")
    print(f"  Silhouette:        {sil:.4f}")
    print(f"  Cosine intra-cl:   {cos_intra:.4f}")
    print(f"  ARI:               {ari:.4f}")
    print(f"  NMI:               {nmi:.4f}")
    print(f"  Homogeneity:       {hom:.4f}")
    print(f"  Completeness:      {comp:.4f}")
    print(f"  V-measure:         {vmeas:.4f}")
    
    return {
        'Varianta': label,
        'Clustere': n_clusters,
        'Noise %': round(noise_pct, 1),
        'Silhouette': round(sil, 4),
        'Cosine intra': round(cos_intra, 4),
        'ARI': round(ari, 4),
        'NMI': round(nmi, 4),
        'Homogeneity': round(hom, 4),
        'Completeness': round(comp, 4),
        'V-measure': round(vmeas, 4)
    }

results_full = []

results_full.append(compute_all_metrics(emb_v1, umap_v1, labels_v1, labels_true, 'V1: Text complet'))
results_full.append(compute_all_metrics(emb_v2, umap_v2, labels_v2, labels_true, 'V2: Fara cuvinte solutie'))
results_full.append(compute_all_metrics(emb_v3, umap_v3, labels_v3, labels_true, 'V3: Anonimizat + fara solutie'))

df_metrics_full = pd.DataFrame(results_full)
print("\nTABEL COMPARATIV")
print(df_metrics_full.to_string(index=False))
df_metrics_full.to_csv('/kaggle/working/metrici_complete.csv', index=False)
import plotly.graph_objects as go

metrici = ['Silhouette', 'Cosine intra', 'ARI', 'NMI', 'Homogeneity', 'Completeness', 'V-measure']
culori = ['#3498db', '#2ecc71', '#e74c3c']
variante = [r['Varianta'] for r in results_full]

fig = go.Figure()

for i, row in enumerate(results_full):
    fig.add_trace(go.Bar(
        name=row['Varianta'],
        x=metrici,
        y=[row[m] for m in metrici],
        marker_color=culori[i],
        text=[f"{row[m]:.3f}" for m in metrici],
        textposition='outside'
    ))

fig.update_layout(
    title='Comparatie metrici clustering',
    xaxis_title='Metrica',
    yaxis_title='Valoare',
    yaxis=dict(range=[0, 1.1]),
    barmode='group',
    width=1100,
    height=600,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)

fig.write_html('/kaggle/working/metrici_comparatie.html')
fig.show()
print("Salvat metrici_comparatie.html")

Varianta: V1: Text complet
  Clustere:          18
  Noise %:           1.3%
  Silhouette:        0.3678
  Cosine intra-cl:   0.9931
  ARI:               0.2339
  NMI:               0.4064
  Homogeneity:       0.5877
  Completeness:      0.3106
  V-measure:         0.4064
Varianta: V2: Fara cuvinte solutie
  Clustere:          28
  Noise %:           9.6%
  Silhouette:        0.6288
  Cosine intra-cl:   0.9928
  ARI:               0.1436
  NMI:               0.3564
  Homogeneity:       0.6930
  Completeness:      0.2399
  V-measure:         0.3564
Varianta: V3: Anonimizat + fara solutie
  Clustere:          28
  Noise %:           4.5%
  Silhouette:        0.5536
  Cosine intra-cl:   0.9934
  ARI:               0.1523
  NMI:               0.3703
  Homogeneity:       0.7105
  Completeness:      0.2504
  V-measure:         0.3703

TABEL COMPARATIV
                     Varianta  Clustere  Noise %  Silhouette  Cosine intra    ARI    NMI  Homogeneity  Completeness  V-measure
             V1

Salvat metrici_comparatie.html


In [14]:
import plotly.graph_objects as go
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from plotly.subplots import make_subplots

STOPWORDS_JURIDICE = [
    'cetatean', 'roman', 'cetatean roman', 'roman cu', 'cu domiciliul',
    'domiciliul', 'domiciliul in', 'in bucuresti', 'bucuresti', 'sector',
    'sectorul', 'seria', 'cnp', 'posesor', 'posesoare', 'posesor al',
    'bi seria', 'ci seria', 'bi', 'ci', 'nr', 'str', 'bl', 'sc', 'et',
    'ap', 'bloc', 'blocul', 'scara', 'etaj', 'etajul', 'apartament',
    'apartamentul', 'doamna', 'domnul', 'strada', 'bulevardul', 'bd',
    'sos', 'calea', 'intrarea', 'piata', 'aleea', 'splaiul',
    'notificarea', 'notificarii', 'notificatorul', 'notificatoarea',
    'executorul', 'judecatoresc', 'inregistrata', 'inregistrate',
    'privind', 'solutionarea', 'dosarului', 'intocmit', 'baza',
    'legii', 'lege', 'legea', 'art', 'alin', 'temeiul', 'conformitate',
    'dispozitie', 'dispozitiei', 'primarului', 'general', 'primar',
    'municipiului', 'municipiului bucuresti', 'primaria', 'emitent',
    'avand', 'vedere', 'avand in vedere', 'in vedere', 'vazand',
    'actele', 'dosarului anexate', 'se', 'din', 'in', 'la', 'de',
    'cu', 'si', 'pe', 'al', 'ale', 'din care', 'care', 'prin',
    'pentru', 'sau', 'nu', 'este', 'fost', 'fiind', 'astfel',
    'solutie', 'per anon', 'addr dom', 'per_anon', 'addr_dom',
]

solutii = ['Compensare/Despagubiri', 'Respins/Negativ', 'Restituire']

def make_heatmap(labels, text_col, title, filename):
    # TF-IDF keywords
    cluster_docs = {}
    for cl in sorted(set(labels)):
        if cl == -1:
            continue
        mask = labels == cl
        cluster_docs[cl] = ' '.join(df_final.loc[mask, text_col].tolist())

    cluster_ids = sorted(cluster_docs.keys())
    corpus = [cluster_docs[cl] for cl in cluster_ids]

    vec = TfidfVectorizer(
        max_features=50000,
        min_df=2,
        max_df=0.85,
        ngram_range=(1, 2),
        stop_words=STOPWORDS_JURIDICE
    )
    tfidf = vec.fit_transform(corpus)
    features = vec.get_feature_names_out()

    keywords = {}
    for i, cl in enumerate(cluster_ids):
        scores = tfidf[i].toarray().flatten()
        top_idx = scores.argsort()[::-1][:5]
        keywords[cl] = [features[j] for j in top_idx]

    # crosstab + procente
    ct = pd.crosstab(labels, df_final['solutie_pdf_grup'])
    ct = ct[ct.index >= 0]
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct['n_docs'] = ct.sum(axis=1)
    ct_pct = ct_pct.sort_values('Restituire', ascending=False)

    z = ct_pct[solutii].values
    cluster_ids_sorted = ct_pct.index.tolist()
    n_docs = ct_pct['n_docs'].values

    y_labels = [
        f"Cl {cl} (n={int(n_docs[i])}) | {', '.join(keywords.get(cl, []))}"
        for i, cl in enumerate(cluster_ids_sorted)
    ]

    fig = go.Figure(data=go.Heatmap(
        z=z,
        x=solutii,
        y=y_labels,
        colorscale='RdYlGn',
        text=np.round(z, 1),
        texttemplate='%{text}%',
        textfont={"size": 10},
        colorbar=dict(title='%'),
        zmin=0,
        zmax=100
    ))

    fig.update_layout(
        title=title,
        xaxis_title='Solutie',
        yaxis_title='Cluster',
        width=1400,
        height=900,
        template='plotly_white',
        margin=dict(l=500)
    )

    fig.write_html(f'/kaggle/working/{filename}.html')
    print(f"Salvat: {filename}.html")

make_heatmap(labels_v1, 'text_v1', 'V1: Text complet - Distributia solutiilor per cluster', 'heatmap_v1')
make_heatmap(labels_v2, 'text_v2', 'V2: Fara cuvinte solutie - Distributia solutiilor per cluster', 'heatmap_v2_final')
make_heatmap(labels_v3, 'text_v3', 'V3: Anonimizat + fara solutie - Distributia solutiilor per cluster', 'heatmap_v3_final')

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning:

Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['addr', 'anexate', 'anon', 'dom', 'per'] not in stop_words.



Salvat: heatmap_v1.html


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning:

Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['addr', 'anexate', 'anon', 'dom', 'per'] not in stop_words.



Salvat: heatmap_v2_final.html


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning:

Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['addr', 'anexate', 'anon', 'dom', 'per'] not in stop_words.



Salvat: heatmap_v3_final.html


In [15]:
df_metrics_full = pd.read_csv('/kaggle/working/metrici_clustering.csv')

# tabel plotly
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_metrics_full.columns),
        fill_color='#2c3e50',
        font=dict(color='white', size=12),
        align='center'
    ),
    cells=dict(
        values=[df_metrics_full[col] for col in df_metrics_full.columns],
        fill_color=[['#ecf0f1', '#ffffff'] * len(df_metrics_full)],
        align='center',
        font=dict(size=11)
    )
)])

fig_table.update_layout(
    title='Tabel comparativ metrici clustering',
    width=1200,
    height=250,
    template='plotly_white'
)

fig_table.write_html('/kaggle/working/tabel_metrici.html')
fig_table.show()
print("Salvat: tabel_metrici.html")

Salvat: tabel_metrici.html
